# Train Passenger Source Simulation

In this notebook, you will learn about passenger generation and queuing:

1. Create passenger source agents for entry stations
2. Generate passengers based on peak/off-peak hours
3. Queue passengers at stations
4. Publish passenger arrival events to MQTT
5. Visualize passenger queues on a map

This demonstrates how passenger demand varies throughout the day and creates load on the train system.

## Step 1: Load Configuration and Setup

Load the train network configuration and connect to MQTT.

In [ ]:
from simulated_city.config import load_config
from simulated_city.mqtt import MqttConnector, MqttPublisher
from simulated_city.agents import Station, SimulationState, PassengerSourceAgent
from datetime import datetime

# Load configuration
config = load_config()

# Connect to MQTT
connector = MqttConnector(config.mqtt, client_id_suffix="passenger-source")
connector.connect()

if connector.wait_for_connection(timeout=5):
    print("✓ Connected to MQTT broker")
    publisher = MqttPublisher(connector)
else:
    print("✗ Failed to connect to MQTT broker")

# Display passenger flow configuration
print("\nPassenger Flow Configuration:")
print(f"Off-peak rate: {config.train_network.passenger_flow.off_peak_passengers_per_10min} passengers/10min")
print(f"\nPeak hours:")
for peak in config.train_network.passenger_flow.peak_hours:
    print(f"  {peak.start}:00 - {peak.end}:00 → {peak.passengers_per_10min} passengers/10min")

## Step 2: Create Simulation State

The simulation state tracks all passengers waiting at stations.

In [ ]:
# Initialize simulation state
sim_state = SimulationState(current_time=datetime.now())

print(f"Simulation state initialized at {sim_state.current_time.strftime('%H:%M:%S')}")
print(f"Active trains: {sim_state.active_train_count}")
print(f"Waiting passengers: {sim_state.total_waiting_passengers}")

## Step 3: Create Station and Passenger Source Agents

We create passenger source agents for each entry station. These agents generate passengers according to the time of day.

In [ ]:
# Convert config to Station objects
route = []
for station_config in config.train_network.route:
    station = Station(
        name=station_config.name,
        station_type=station_config.type,
        location_lat=station_config.location.lat,
        location_lon=station_config.location.lon,
        exit_percentage=station_config.exit_percentage,
    )
    route.append(station)

# Separate entry and exit stations
entry_stations = [s for s in route if s.station_type == "entry"]
exit_stations = [s for s in route if s.station_type == "exit"]

print(f"Entry stations: {len(entry_stations)}")
for station in entry_stations:
    print(f"  - {station.name}")

print(f"\nExit stations: {len(exit_stations)}")
for station in exit_stations:
    print(f"  - {station.name}")

In [ ]:
# Create passenger source agents for each entry station
passenger_sources = []

for entry_station in entry_stations:
    source = PassengerSourceAgent(
        station=entry_station,
        exit_stations=exit_stations,
        passenger_flow_config=config.train_network.passenger_flow,
        mqtt_publisher=publisher,
        mqtt_base_topic=config.train_network.mqtt_base_topic,
    )
    passenger_sources.append(source)

print(f"Created {len(passenger_sources)} passenger source agents")

## Step 4: Simulate Passenger Arrivals During Peak Hours

Let's simulate passenger arrivals during peak hours (18:00, 19:00, 20:00) and see how queues build up.

**Key concept**: During peak hours, many more passengers arrive, creating demand for extra trains.

In [ ]:
import time

# Simulate three peak hours: 18:00, 19:00, 20:00
peak_hours = [18, 19, 20]

print("Simulating passenger arrivals during peak hours...\n")

for hour in peak_hours:
    # Set simulation time to this hour
    sim_time = datetime.now().replace(hour=hour, minute=0, second=0)
    sim_state.current_time = sim_time
    
    print(f"=== Hour: {hour}:00 ===")
    
    for source in passenger_sources:
        # Get arrival rate for this hour
        rate = source.get_arrival_rate(hour)
        
        # Generate passengers
        passengers = source.generate_passengers(rate)
        
        # Add to station queue
        queue = sim_state.get_station_queue(source.station.name)
        queue.add_passengers(passengers)
        sim_state.total_passengers_generated += len(passengers)
        
        # Publish arrival event
        source.publish_arrival(len(passengers), queue.count)
        
        print(f"  {source.station.name}:")
        print(f"    New arrivals: {len(passengers)} passengers")
        print(f"    Queue size: {queue.count} passengers")
        print(f"    Avg wait time: {queue.average_wait_time_seconds:.1f}s")
    
    print()
    time.sleep(0.5)  # Brief pause between hours

print(f"\n✓ Total passengers generated: {sim_state.total_passengers_generated}")
print(f"✓ Total waiting: {sim_state.total_waiting_passengers}")

## Step 5: Inspect Queue Details

Let's examine the waiting passengers at each station in detail.

In [ ]:
print("Current Queue Status:\n")

for station_name, queue in sim_state.station_queues.items():
    print(f"{station_name}:")
    print(f"  Waiting: {queue.count} passengers")
    print(f"  Avg wait time: {queue.average_wait_time_seconds:.1f}s")
    
    # Show exit station distribution
    if queue.count > 0:
        exit_counts = {}
        for passenger in queue.waiting_passengers[:10]:  # Sample first 10
            exit_station = passenger.exit_station
            exit_counts[exit_station] = exit_counts.get(exit_station, 0) + 1
        
        print(f"  Exit destinations (sample):")
        for exit_station, count in exit_counts.items():
            print(f"    → {exit_station}: {count} passengers")
    print()

## Step 6: Visualize Passenger Queues on Map

We can visualize the passenger queues on a map, where larger circles represent more waiting passengers.

In [ ]:
import json

# Create GeoJSON for passenger queues
features = []

for station in entry_stations:
    queue = sim_state.get_station_queue(station.name)
    
    feature = {
        "type": "Feature",
        "geometry": {
            "type": "Point",
            "coordinates": [station.location_lon, station.location_lat]
        },
        "properties": {
            "name": station.name,
            "type": "entry",
            "waiting_count": queue.count,
            "avg_wait_time": round(queue.average_wait_time_seconds, 1),
            "circle_radius": min(queue.count / 10, 50),  # Scale circle size
        }
    }
    features.append(feature)

geojson = {
    "type": "FeatureCollection",
    "features": features
}

print("GeoJSON for passenger queues:")
print(json.dumps(geojson, indent=2))

## Step 7: Compare Peak vs Off-Peak Arrivals

Let's compare passenger generation rates between peak and off-peak hours.

In [ ]:
# Test one passenger source
test_source = passenger_sources[0]

peak_hour = 19
off_peak_hour = 14

peak_rate = test_source.get_arrival_rate(peak_hour)
off_peak_rate = test_source.get_arrival_rate(off_peak_hour)

print(f"Passenger Arrival Rates for {test_source.station.name}:")
print(f"  Peak hour ({peak_hour}:00): {peak_rate} passengers/10min")
print(f"  Off-peak hour ({off_peak_hour}:00): {off_peak_rate} passengers/10min")
print(f"\n  Peak-to-off-peak ratio: {peak_rate / off_peak_rate:.1f}x")

## Cleanup

In [ ]:
# Disconnect from MQTT
if connector.client and connector.client.is_connected():
    connector.disconnect()
    print("✓ Disconnected from MQTT broker")

## Exercise: Experiment with Arrival Patterns

Try these experiments:

1. **Modify peak hours**: Edit `config.yaml` to add an early morning peak (7-9 AM)
2. **Create a super-peak event**: Manually generate 500 passengers at one station to simulate a concert or sports event
3. **Track queue growth**: Run passenger generation in a loop for 60 simulated minutes and plot queue size over time

Example for experiment 2:
```python
# Simulate event at Entry Station 1
event_station = entry_stations[0]
source = passenger_sources[0]

event_passengers = source.generate_passengers(500)
queue = sim_state.get_station_queue(event_station.name)
queue.add_passengers(event_passengers)

print(f"Event! {len(event_passengers)} passengers arrived at {event_station.name}")
print(f"Queue now: {queue.count} passengers")
```

## Next Steps

Continue to **05_train_control_center.ipynb** to learn how the control center monitors these queues and dispatches extra trains when needed.